In [1]:
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr 
from scipy.stats import linregress
from datetime import datetime, timedelta
import pandas as pd
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import os
import argparse
import xclim


In [5]:
xr.open_dataset("/esarchive/obs/ecad/eobs_v29/daily_mean/tasmax/tasmax_196802.nc")

<xarray.Dataset> Size: 11MB
Dimensions:    (time: 29, latitude: 201, longitude: 464)
Coordinates:
  * latitude   (latitude) float64 2kB 25.38 25.62 25.88 ... 74.88 75.12 75.38
  * longitude  (longitude) float64 4kB -40.38 -40.12 -39.88 ... 75.12 75.38
  * time       (time) datetime64[ns] 232B 1968-02-01 1968-02-02 ... 1968-02-29
Data variables:
    tasmax     (time, latitude, longitude) float32 11MB ...
Attributes:
    CDI:                        Climate Data Interface version 1.9.8 (https:/...
    Conventions:                CF-1.4
    E-OBS_version:              29.0e
    References:                 http://surfobs.climate.copernicus.eu/dataacce...
    CDO:                        Climate Data Operators version 1.9.8 (https:/...
    history_of_appended_files:  Thu May  2 13:37:08 2024: Appended file /esar...
    nco_openmp_thread_number:   1
    history:                    Thu May  2 13:37:09 2024: ncatted -O -a units...
    NCO:                        4.7.3

## Merge varaibles in one file

In [7]:
import xarray as xr
import os

# --- 1. Configuration ---

# The folder containing your NetCDF files
input_directory = "/gpfs/projects/bsc32/bsc167965/observational_TX"

# The sites you want to process
sites = ["cordoba","hannover", "stockholm", "lyon", "belgrado", "marrakech"]

# Target variable names in the final file
variables = ["prlr", "tas", "tasmin", "tasmax"]
# Variable names in the original source files
variables_file = ["prlr", "tas", "tasmin", "TX"]

# --- 2. Main Processing Loop ---
print("Starting to merge NetCDF files with strictly enforced string units...")

for site in sites:
    print(f"\nProcessing site: {site.capitalize()}")
    
    datasets_to_merge = []
    
    try:
        # Loop through each variable to load, process, and assign units
        for var_name, var_name_file in zip(variables, variables_file):
            
            file_path = os.path.join(input_directory, f"observational_{var_name_file}_data_{site}_1950_2024.nc")
            
            # Open the dataset      
            ds = xr.open_dataset(file_path)
            
            # --- Handle each variable to ensure correct name and string units ---
            
            if var_name == "prlr":
                print(f"  - Converting 'prlr' to daily 'pr'...")
                conversion_factor = 86400000
                ds['prlr'] = ds['prlr'] * conversion_factor
                ds = ds.rename({'prlr': 'pr'})
                # Explicitly set units as a string
                ds['pr'].attrs['units'] = str('mm/day') 
                ds['pr'].attrs['long_name'] = 'Daily Total Precipitation'

            elif var_name == "tasmax":
                print(f"  - Renaming 'TX' to 'tasmax' and setting units...")
                # Explicitly set units as a string
                ds['tasmax'].attrs['units'] = str('K')
                ds['tasmax'].attrs['long_name'] = 'Daily Maximum Temperature'

            else: # This handles 'tas' and 'tasmin'
                print(f"  - Ensuring units for '{var_name}' are set...")
                # Explicitly set units as a string
                ds[var_name].attrs['units'] = str('K')
                if var_name == 'tas':
                    ds['tas'].attrs['long_name'] = 'Daily Mean Temperature'
                elif var_name == 'tasmin':
                    ds['tasmin'].attrs['long_name'] = 'Daily Minimum Temperature'

            datasets_to_merge.append(ds)

        # Merge all the datasets for the site into one
        combined_ds = xr.merge(datasets_to_merge)
        
        output_filename = f"combined_obs_eobs_data_{site}.nc"
        output_path = os.path.join(input_directory, output_filename)
        
        # Save the merged dataset, overwriting old files
        combined_ds.to_netcdf(output_path)
        
        print(f"✅ Success! Combined file with guaranteed string units saved to: {output_path}")

    except Exception as e:
        print(f"❌ An unexpected error occurred for site '{site}': {e}")

print("\nAll sites processed successfully!")

Starting to merge NetCDF files with strictly enforced string units...

Processing site: Cordoba
  - Converting 'prlr' to daily 'pr'...
  - Ensuring units for 'tas' are set...
  - Ensuring units for 'tasmin' are set...
  - Renaming 'TX' to 'tasmax' and setting units...
✅ Success! Combined file with guaranteed string units saved to: /gpfs/projects/bsc32/bsc167965/observational_TX/combined_obs_eobs_data_cordoba.nc

Processing site: Hannover
  - Converting 'prlr' to daily 'pr'...
  - Ensuring units for 'tas' are set...
  - Ensuring units for 'tasmin' are set...
  - Renaming 'TX' to 'tasmax' and setting units...
✅ Success! Combined file with guaranteed string units saved to: /gpfs/projects/bsc32/bsc167965/observational_TX/combined_obs_eobs_data_hannover.nc

Processing site: Stockholm
  - Converting 'prlr' to daily 'pr'...
  - Ensuring units for 'tas' are set...
  - Ensuring units for 'tasmin' are set...
  - Renaming 'TX' to 'tasmax' and setting units...
✅ Success! Combined file with guarant

## Code to compute spei

In [34]:
import xarray as xr
import xclim.indices as xcli
import numpy as np
import os

# --- 1. General Configuration ---

# Directory where your 'combined_climate_data' files are stored
data_directory = "/gpfs/projects/bsc32/bsc167965/observational_TX"

# List of sites to process
sites = ["cordoba", "hannover", "stockholm", "lyon", "belgrado"]

site_latitudes = {
    "cordoba": 37.88,
    "hannover": 52.37,
    "stockholm": 59.33,
    "lyon": 45.76,
    "belgrado": 44.78,
    "marrakech": 31.63
}

# --- 2. SPEI Calculation Parameters ---

# Choose your PET calculation method: 'thorn' or 'hg'
method = "hg" 

# Define the SPEI timescale in months (e.g., 3, 6, 12)

# 3-month time-scale: considered the best timescale for capturing moisture conditions in the topsoil layer

scale = 3


# Define the reference period for calibrating the index
ref_years = np.arange(1981, 2011)


# --- 3. Main Processing Loop ---

print(f" Starting SPEI calculation for {len(sites)} sites...")
print(f"   Method: {method.upper()}, Timescale: {scale}-month")

for site in sites:
    print(f"\n--- Processing: {site.capitalize()} ---")
    
    try:
        # Construct the input filename for the current site
        input_filename = f"combined_obs_eobs_data_{site}.nc"
        input_path = os.path.join(data_directory, input_filename)
        
        # Load the combined dataset for the site
        climate_data = xr.open_dataset(input_path)
        print(f"  -> Successfully loaded {input_filename}")

        site_lat = xr.DataArray(site_latitudes[site], name="lat", attrs={"units": ""})

        # Define the unique name for the new SPEI variable
        indx_name = f"spei_{method}_{scale}"

        # --- Calculate Potential Evapotranspiration (PET) ---
        print(f"  -> Calculating Potential Evapotranspiration (PET) using '{method}' method...")
        if method == "thorn":
            # simplest, less accurate emthod 
            climate_data['pet'] = xcli.potential_evapotranspiration(
                tas=climate_data.tas, 
                method="thornthwaite",
                lat=site_lat
            )
        elif method == "hg":
            # method for when tasmax and tasmin are available. Estimates solar radiation with 
            climate_data['pet'] = xcli.potential_evapotranspiration(
                tas=climate_data.tas,
                tasmin=climate_data.tasmin,
                tasmax=climate_data.tasmax,
                method="HG85",
                lat=site_lat
            )
        else:
            print(f"  -> ERROR: Method '{method}' not recognized! Skipping site.")
            continue

        # --- Calculate Climatic Water Balance (WB) ---
        print("  -> Calculating climatic water balance...")
        wb = xcli.water_budget(pr=climate_data.pr, evspsblpot=climate_data.pet)

        wb_monthly = wb.resample(time='MS').sum()

        # --- Calculate SPEI ---
        print(f"  -> Calculating {scale}-month SPEI...")
        wb_calibration = wb.sel(time=wb.time.dt.year.isin(ref_years))

        # Computes the baseline for the entire period
        climate_data[indx_name] = xcli.standardized_precipitation_evapotranspiration_index(
            wb_monthly,
            freq='MS',
            window=scale,
            dist='fisk' # Or 'gamma'
        )

        climate_data[indx_name] = climate_data[indx_name].ffill(dim='time')
                
        # --- Filter and Save the Desired Variables ---
        print(f"  -> Filtering dataset to keep 'tasmax' and '{indx_name}'...")
        
        # Select only the variables you want to keep
        output_ds = climate_data[['tasmax', indx_name]]
        
        # Define a new, descriptive output filename
        output_filename = f"spei_tasmax_{site}_{method}_{scale}.nc"
        output_path = os.path.join(data_directory, output_filename)
        
        # Save the filtered dataset
        output_ds.to_netcdf(output_path)
        print(f"✅ Success! Filtered data saved to: {output_path}")

    except FileNotFoundError:
        print(f"  -> ERROR: File not found at {input_path}. Skipping this site.")
    except Exception as e:
        print(f"  -> An unexpected error occurred for site '{site}': {e}")

print("\nAll sites processed. ✅")

 Starting SPEI calculation for 5 sites...
   Method: HG, Timescale: 3-month

--- Processing: Cordoba ---
  -> Successfully loaded combined_obs_eobs_data_cordoba.nc
  -> Calculating Potential Evapotranspiration (PET) using 'hg' method...
  -> Calculating climatic water balance...
  -> Calculating 3-month SPEI...


/home/bsc/bsc167965/.conda/envs/arnaugmenv/lib/python3.9/site-packages/xclim/indices/_agro.py:1306: UserWarning: Inputting an offset will be deprecated in xclim>=0.50.0. 
  warnings.warn("Inputting an offset will be deprecated in xclim>=0.50.0. ")


  -> Filtering dataset to keep 'tasmax' and 'spei_hg_3'...
✅ Success! Filtered data saved to: /gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_cordoba_hg_3.nc

--- Processing: Hannover ---
  -> Successfully loaded combined_obs_eobs_data_hannover.nc
  -> Calculating Potential Evapotranspiration (PET) using 'hg' method...
  -> Calculating climatic water balance...
  -> Calculating 3-month SPEI...


/home/bsc/bsc167965/.conda/envs/arnaugmenv/lib/python3.9/site-packages/xclim/indices/_agro.py:1306: UserWarning: Inputting an offset will be deprecated in xclim>=0.50.0. 
  warnings.warn("Inputting an offset will be deprecated in xclim>=0.50.0. ")


  -> Filtering dataset to keep 'tasmax' and 'spei_hg_3'...
✅ Success! Filtered data saved to: /gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_hannover_hg_3.nc

--- Processing: Stockholm ---
  -> Successfully loaded combined_obs_eobs_data_stockholm.nc
  -> Calculating Potential Evapotranspiration (PET) using 'hg' method...
  -> Calculating climatic water balance...
  -> Calculating 3-month SPEI...


...

  -> Filtering dataset to keep 'tasmax' and 'spei_hg_3'...
✅ Success! Filtered data saved to: /gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_stockholm_hg_3.nc

--- Processing: Lyon ---
  -> Successfully loaded combined_obs_eobs_data_lyon.nc
  -> Calculating Potential Evapotranspiration (PET) using 'hg' method...
  -> Calculating climatic water balance...
  -> Calculating 3-month SPEI...


/home/bsc/bsc167965/.conda/envs/arnaugmenv/lib/python3.9/site-packages/xclim/indices/_agro.py:1306: UserWarning: Inputting an offset will be deprecated in xclim>=0.50.0. 
  warnings.warn("Inputting an offset will be deprecated in xclim>=0.50.0. ")


  -> Filtering dataset to keep 'tasmax' and 'spei_hg_3'...
✅ Success! Filtered data saved to: /gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_lyon_hg_3.nc

--- Processing: Belgrado ---
  -> Successfully loaded combined_obs_eobs_data_belgrado.nc
  -> Calculating Potential Evapotranspiration (PET) using 'hg' method...
  -> Calculating climatic water balance...
  -> Calculating 3-month SPEI...


/home/bsc/bsc167965/.conda/envs/arnaugmenv/lib/python3.9/site-packages/xclim/indices/_agro.py:1306: UserWarning: Inputting an offset will be deprecated in xclim>=0.50.0. 
  warnings.warn("Inputting an offset will be deprecated in xclim>=0.50.0. ")


  -> Filtering dataset to keep 'tasmax' and 'spei_hg_3'...
  -> An unexpected error occurred for site 'belgrado': [Errno 13] Permission denied: '/gpfs/projects/bsc32/bsc167965/observational_TX/spei_tasmax_belgrado_hg_3.nc'

All sites processed. ✅


In [22]:
xr.open_dataset('/gpfs/projects/bsc32/bsc167965/observational_TX/combined_obs_eobs_data_hannover.nc')['tasmin'].values[-20:]

array([277.19125, 277.5969 , 275.06314, 275.35687, 278.5644 , 277.69376,
       276.4675 , 277.02625, 277.1819 , 276.66315, 275.26126, 274.44   ,
       280.785  , 281.9025 , 276.3375 , 273.23438, 280.09564, 279.60687,
       277.8081 , 277.665  ], dtype=float32)

In [23]:

print(f"  -> Checking results for '{site.capitalize()}'...")
print("First 12 values:")
print(climate_data[indx_name].head(12))
print("\nLast 5 values:")
print(climate_data[indx_name].tail(5))


  -> Checking results for 'Belgrado'...
First 12 values:
<xarray.DataArray 'spei_hg_3' (time: 12)> Size: 96B
array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])
Coordinates:
  * time     (time) datetime64[ns] 96B 1950-01-01 1950-01-02 ... 1950-01-12
Attributes:
    calibration_period:  ('1950-01-01', '2023-12-01')
    freq:                MS
    window:              3
    scipy_dist:          fisk
    method:              ML
    group:               time.month
    units:               1
    time_indexer:        {}

Last 5 values:
<xarray.DataArray 'spei_hg_3' (time: 5)> Size: 40B
array([nan, nan, nan, nan, nan])
Coordinates:
  * time     (time) datetime64[ns] 40B 2023-12-27 2023-12-28 ... 2023-12-31
Attributes:
    calibration_period:  ('1950-01-01', '2023-12-01')
    freq:                MS
    window:              3
    scipy_dist:          fisk
    method:              ML
    group:               time.month
    units:               1
    time_indexer:        {}
